# **Pull GitHub Repository**

In [ ]:
!pip install -q torchmetrics timm

In [ ]:
from google.colab import userdata

!rm -rf /content/ECM3401_Individual_Project

token = userdata.get('GitHub')
!git clone -b heavy_decoder_change https://{token}@github.com/sccthomas/ECM3401_Individual_Project.git

In [ ]:
import sys

sys.path.append('/content/ECM3401_Individual_Project/')
!ls /content/ECM3401_Individual_Project/src/

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# **Define the Model**

In [1]:
import torch
from src.vision_transformer.model import SemanticSegmentationVisionTransformer

# --------------------------------------------
# Parameters
# --------------------------------------------

device = torch.device("mps")
metric_device = torch.device("cpu")

image_dims = (3, 256, 256)  # Input image dimensions
patch_embedding_scale_1 = (16, 768)  # Patch size and embedding dimension for scale 1
patch_embedding_scale_2 = (8, 512)  # Patch size and embedding dimension for scale 2
patch_embedding_scale_3 = (4, 256)  # Patch size and embedding dimension for scale 3

# --------------------------------------------
# Model Initialization
# --------------------------------------------
model = SemanticSegmentationVisionTransformer(
    # - Image dimensions
    image_dims=image_dims,
    # - Hyper Parameters
    num_encoder_layers=6,
    use_heavyweight_decoder=True,
    use_swin_transformer=False,
    use_skip_layer_gated_attention=False,
    use_learnable_skip_layers=True,
    skip_layer_ratio=2,
    encoder_dropout_rate=0.2,
    patch_fusion_dropout_rate=0.2,
    decoder_dropout_rate=0.2,
    num_encoder_heads=8,
    num_classes=1,
    # - Scales
    patch_embedding_scale_1=patch_embedding_scale_1,
    patch_embedding_scale_2=patch_embedding_scale_2,
    patch_embedding_scale_3=patch_embedding_scale_3,
).to(device)

In [ ]:
model

### **Optional Load State**

In [ ]:
import torch

# Load the model's state_dict
path = "/content/model.pth"
checkpoint = torch.load(path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint)

# **Train the Model**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from src.dataset.snow import SnowDataset
from src.training.train import train_model
import os as _os

# --------------------------------------------
# Environment
# --------------------------------------------
_os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --------------------------------------------
# Parameters
# --------------------------------------------
dataset_dir = "/content/drive/MyDrive/snow_dataset"  # Replace with your dataset path
batch_size = 32
num_epochs = 25
learning_rate = 1e-4
patience = 7  # Early stopping patience

# --------------------------------------------
# Dataset and DataLoader
# --------------------------------------------

# Load the dataset and split it into train and validation sets
train_dataset = SnowDataset(
    dataset_dir_path=dataset_dir,
    resize=True,
    normalize=True,
    rotate=True,
    augment_image=True,
)
print("Length of dataset:", len(train_dataset))
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size], generator=generator)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=12,
    persistent_workers=True,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=12,
    persistent_workers=True,
    pin_memory=True,
)

# --------------------------------------------
# Loss, Optimizer, and Scheduler
# --------------------------------------------
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=num_epochs // 2, T_mult=1)

# --------------------------------------------
# Mixed Precision Setup
# --------------------------------------------
scaler = torch.amp.GradScaler(device.type)

# --------------------------------------------
# Train Model
# --------------------------------------------
train_model(
    model=model,
    num_epochs=num_epochs,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    train_loader=train_loader,
    val_loader=val_loader,
    patience=patience,
    device=device,
    metric_device=metric_device,
)

In [ ]:
!cp /content/best_model.pth /content/drive/MyDrive/best_model.pth

# **Evaluate the Model**

In [2]:
parent_dir = "/Users/samuelthomas/Documents/University/4thYr_Final/ECM3401_Individual_Literature_Review_and_Project/SNOW_Semantic_Segmentation/trained_models/supervised/"

In [3]:
%matplotlib inline

In [4]:
import torch

# pre_or_post = "post"  # "post" or "pre"

# Load the model's state_dict
# path = f"{parent_dir}/model_{pre_or_post}_fine_tune.pth"
path = f"{parent_dir}/model.pth"
checkpoint = torch.load(path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint)
model = model.eval()

## **Causality**

In [5]:
from src.dataset.snow import SnowDataset
from torch.utils.data import DataLoader

dataset_dir = "/Users/samuelthomas/Documents/University/4thYr_Final/ECM3401_Individual_Literature_Review_and_Project/SNOW_Semantic_Segmentation/snow_dataset"
# dataset_dir = "/content/drive/MyDrive/snow_dataset"

# Dataset and DataLoader
batch_size = 5

train_dataset = SnowDataset(
    dataset_dir_path=dataset_dir,
    len_override=30,
    resize=True,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=1,
)
images, masks = next(iter(train_loader))
images, masks = images.to(device), masks.to(device)

In [6]:
import matplotlib.pyplot as plt
from src.training.evaluation import (
    # evaluate_with_illumination_modifications,
    # evaluate_with_color_jitter,
    # evaluate_with_noise_addition,
    # evaluate_with_blur,
    # evaluate_with_synthetic_background,
    # evaluate_with_stain_variation,
    evaluate_with_no_modifications
)

# path = f"{parent_dir}/figures/{pre_or_post}_fine_tune"
path = f"{parent_dir}/figures"

for idx in range(batch_size):
    evaluate_with_no_modifications(
        model,
        images[idx],
        masks[idx],
        device,
        path=path,
        name=f"image_{idx + 1}",
    )
    plt.close('all')
    # for func in [
    #     evaluate_with_illumination_modifications,
    #     evaluate_with_color_jitter,
    #     evaluate_with_noise_addition,
    #     evaluate_with_blur,
    #     evaluate_with_synthetic_background,
    #     evaluate_with_stain_variation,
    # ]:
    #     func(
    #         model,
    #         images[idx],
    #         masks[idx],
    #         device,
    #         path=path,
    #         name=f"image_{idx + 1}",
    #         background_and_cells=False,
    #     )
    #     func(
    #         model,
    #         images[idx],
    #         masks[idx],
    #         device,
    #         path=path,
    #         name=f"image_{idx + 1}",
    #         background_and_cells=True,
    #     )
    #     plt.close('all')

## **Unseen Dataset**

In [5]:
from src.dataset.cryonuseg import CryoNuSegDataset
from torch.utils.data import DataLoader
import torch.nn as nn
import src.training.metrics as metrics

# Dataset and DataLoader
dataset_dir = "/Users/samuelthomas/Documents/University/4thYr_Final/ECM3401_Individual_Literature_Review_and_Project/SNOW_Semantic_Segmentation/CryoNuSeg"
batch_size = 30
unseen_dataset = CryoNuSegDataset(
    dataset_dir_path=dataset_dir,
    resize=True,
    normalize=True,
)
unseen_loader = DataLoader(
    unseen_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=1,
)

criterion = nn.BCEWithLogitsLoss()
metric_tracker = metrics.SegmentationMetrics(len_dataset=batch_size, device=metric_device)

In [6]:
images, masks = next(iter(unseen_loader))
images, masks = images.to(device), masks.to(device)

In [7]:
import torch

with torch.amp.autocast(device_type=device.type, dtype=torch.bfloat16):
    outputs = model.forward(images)
    loss = criterion(outputs, masks)

metric_tracker.update_metrics(outputs, masks, loss.item())
metric_tracker.end_of_epoch()
print("------- Unseen Dataset Metrics -------")
print(metric_tracker)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/amp/autocast_mode.py:332: UserWarning: In MPS autocast, but the target dtype is not supported. Disabling autocast.
MPS Autocast only supports dtype of torch.bfloat16 currently.
  warnings.warn(error_message)


RuntimeError: MPS backend out of memory (MPS allocated: 33.01 GB, other allocations: 2.70 GB, max allowed: 36.27 GB). Tried to allocate 2.50 GB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).